# N13. Five seeds per arm, and C against D at every severity

Rebuilt version. Three things changed after the first attempt.

1. **Training now keeps the whole dataset on the GPU.** The first version moved every
   batch from numpy to the device and called `.item()` on every batch, which forced the
   GPU to wait for the CPU 228 times per epoch. On a CPU runtime that cost 13.7 seconds
   per epoch, which is where the 2641 second runs came from. One run should now take
   about 2 minutes on a T4 or L4.
2. **The settings live inside the training cell.** The earlier `NameError: ARMS is not
   defined` happened because changing the runtime type restarts the session and wipes
   every variable. Now a single cell defines the arms and runs them, so that cannot
   happen again.
3. **A timing probe runs first.** It trains one arm, prints the seconds it took and the
   projected total, and stops if the projection is unreasonable.

Why the notebook exists. Every arm in the report was trained once, and differences of
0.33 to 1.71 points then carry claims such as "the erasure output adds nothing". A
single run cannot support a null claim: your first two runs of arm B already differed,
0.1617 against 0.1513 in validation loss, from the seed alone.

It also prints C against D at every severity. The first run showed the split is by
occluder colour, not by severity: D beats C under grey at every severity, C beats D under
black at low severity and is inconclusive above that. Note that the original evaluation
never used the erasure output of C or D, so it compared bit predictions only. The unified
evaluation below scores C and D with their erasure output as designed.

An equivalence margin of 1 point is used. A difference is called equivalent to zero only
when the interval across seeds lies entirely inside plus or minus 1 point. Anything else
is reported as inconclusive, never as "no effect".

**Added in this version**

* White occluders are evaluated alongside grey and black, matching N12.
* A checkpoint check: arms B and M have identical settings, so any gap between them is
  run to run noise. A large gap means some checkpoints came from older training code.
* A unified evaluation: the published decoder (arm P) plus every arm, scored with soft
  decoding, hard decoding with exact fair tie breaking, soft decoding weighted by the
  erasure output (C and D), and soft and hard decoding with the true hidden cell mask
  (oracle ceiling). It also prints the grey minus black gap per arm.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os, sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
if DEV_.type != 'cuda':
    print('\n*** THIS IS A CPU RUNTIME. One training run takes about 40 minutes here')
    print('*** and about 2 minutes on a GPU. Use Runtime, Change runtime type, T4 or L4,')
    print('*** then rerun every cell from the top before training.')
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)
print('statistics ready')


statistics ready


In [7]:
# ---------------------------------------------------------------- Flying ArUco v2
FLY = f'{WORK}/flyingarucov2'
if not os.path.isdir(FLY):
    os.makedirs(FLY, exist_ok=True)
    tar = f'{CACHE}/flyingarucov2.tar.gz'
    assert os.path.exists(tar), f'{tar} missing, see N0'
    subprocess.run(['tar','-xzf',tar,'-C',FLY], check=True)
fly_json = sorted(glob.glob(f'{FLY}/**/*.json', recursive=True))
assert len(fly_json) == N0M['flying_aruco_v2']['images'], \
    f'{len(fly_json)} annotations on disk, N0 manifest says {N0M["flying_aruco_v2"]["images"]}'

def bits_for(mid, rot, conv):
    b = CB[mid]; k = rot if conv in (0,1) else -rot
    return np.rot90(b, k) if conv in (0,2) else np.rot90(b, -k)

ref = NumpyDecoder(Wk); probe = []
for jp in fly_json[:60]:
    img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    for m in json.load(open(jp))['markers'][:4]:
        x32, _ = prepare(img, np.zeros(img.shape[:2], np.uint8), get_corners(m))
        if x32 is None: continue
        probe.append((ref(x32), int(m['id']), int(m.get('rot', m.get('rotation', 0)))))
assert len(probe) > 20, f'only {len(probe)} probe markers, Flying ArUco did not load'
scores = {c: float(np.mean([np.mean((p>0.5).astype(np.uint8)==bits_for(i,r,c)) for p,i,r in probe]))
          for c in range(4)}
CONV = max(scores, key=scores.get)
assert scores[CONV] > 0.90, f'no rotation convention agrees ({scores}), labels would be wrong'
print(f'{len(fly_json)} Flying ArUco v2 annotations | rotation convention {CONV} ({scores[CONV]:.4f})')

2500 Flying ArUco v2 annotations | rotation convention 0 (0.9912)


In [8]:
# ---------------------------------------------------------------- training crops
OCC_TRAIN = [0.0, 0.10, 0.20, 0.25, 0.35, 0.45, 0.55]
N_SAMPLES = 5

def build_training_set(value):
    """value is 127 for arms B, C, D, M, G, or 'random' for arm E."""
    X, Ybit, Yocc = [], [], []
    t0 = time.time()
    for n, jp in enumerate(fly_json):
        img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        key = os.path.basename(jp)
        for m in json.load(open(jp))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            if mid >= 250: continue
            bits = bits_for(mid, int(m.get('rot', m.get('rotation', 0))), CONV)
            rng = np.random.default_rng(zlib.crc32(f'train|{key}|{mid}'.encode()))
            for L in rng.choice(OCC_TRAIN, N_SAMPLES, replace=False):
                v = int(rng.integers(0,256)) if value == 'random' else value
                box = occluder_box(img, cn, float(L), 'edge', key, mid)
                oi, om = apply_occluder(img, box, v)
                x32, occ = prepare(oi, om, cn)
                if x32 is None: continue
                X.append((x32*255).astype(np.uint8)); Ybit.append(bits); Yocc.append(occ)
        if n % 800 == 0: print(f'  {value}: {n}/{len(fly_json)} ({time.time()-t0:.0f}s) crops={len(X)}')
    X = np.stack(X); Ybit = np.stack(Ybit).astype(np.float32); Yocc = np.stack(Yocc).astype(np.float32)
    assert len(X) > 50000, f'only {len(X)} crops, expected about 64840'
    print(f'  {value}: {len(X)} crops, mean occluded cells {Yocc.sum((1,2)).mean():.2f}, '
          f'{time.time()-t0:.0f}s')
    return X, Ybit, Yocc

SETS = {127: build_training_set(127), 'random': build_training_set('random')}
print('\ntraining sets ready')

  127: 0/2500 (0s) crops=40
  127: 800/2500 (19s) crops=20830
  127: 1600/2500 (36s) crops=41645
  127: 2400/2500 (54s) crops=62275
  127: 64840 crops, mean occluded cells 7.00, 56s
  random: 0/2500 (0s) crops=40
  random: 800/2500 (18s) crops=20830
  random: 1600/2500 (36s) crops=41645
  random: 2400/2500 (53s) crops=62275
  random: 64840 crops, mean occluded cells 7.00, 56s

training sets ready


In [9]:
# =================================================================================
# Everything the training loop needs is defined in this one cell, so a runtime
# restart cannot leave half of it missing.
# =================================================================================
CKPT = f'{BASE}/ckpt_n13'; os.makedirs(CKPT, exist_ok=True)

SEEDS   = [0, 1, 2, 3, 4]
MARGIN  = 1.0               # equivalence margin, in accuracy points
EPOCHS, PATIENCE, BS = 200, 20, 256
ALLOW_CPU = False           # set True only if you accept about 40 minutes per run

# arm -> output channels, loss masking, pooling, training occluder brightness
ARMS = {
 'B': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'C': dict(out_ch=2, mask_bits=True,  pool='max', value=127),
 'D': dict(out_ch=2, mask_bits=False, pool='max', value=127),
 'E': dict(out_ch=1, mask_bits=False, pool='max', value='random'),
 'M': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'G': dict(out_ch=1, mask_bits=False, pool='avg', value=127),
}
assert DEV_.type == 'cuda' or ALLOW_CPU, (
    'This is a CPU runtime and 30 runs would take about 20 hours. '
    'Switch to a GPU (Runtime, Change runtime type), rerun every cell from the top, '
    'or set ALLOW_CPU = True if you really mean it.')

GPU_SETS = {}
for v, (X, Yb, Yo) in SETS.items():
    GPU_SETS[v] = (torch.from_numpy(X).to(DEV_),                 # uint8 (N,32,32)
                   torch.from_numpy(Yb).to(DEV_),                # float32 (N,6,6)
                   torch.from_numpy(Yo).to(DEV_))
    print(f'  set {v}: {X.nbytes/1e6:.0f} MB of crops moved to {DEV_}')

SPLIT = {}
for v, (X, _, _) in SETS.items():
    idx = np.random.default_rng(42).permutation(len(X)); cut = int(0.9*len(X))
    SPLIT[v] = (torch.from_numpy(idx[:cut]).to(DEV_), torch.from_numpy(idx[cut:]).to(DEV_))

def train_arm(arm, seed, epochs=EPOCHS, patience=PATIENCE, save=True):
    cfg = ARMS[arm]
    Xg, Bg, Og = GPU_SETS[cfg['value']]
    tr, va = SPLIT[cfg['value']]
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    if save and os.path.exists(path):
        net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
        net.load_state_dict(torch.load(path, map_location=DEV_)); net.eval()
        return net, None, 0.0
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)

    def run(ids, train):
        net.train(train)
        order = ids[torch.randperm(len(ids), device=DEV_)] if train else ids
        tot = torch.zeros((), device=DEV_); n = 0
        with torch.set_grad_enabled(train):
            for s in range(0, len(order), BS):
                b  = order[s:s+BS]
                xb = Xg[b].unsqueeze(1).float()/255.0
                bb = Bg[b]; ob = Og[b]
                if train:                                    # same augmentation as N3b
                    k, fl = int(rng.integers(0,4)), int(rng.integers(0,3))
                    if k:
                        xb = torch.rot90(xb,k,dims=(-2,-1))
                        bb = torch.rot90(bb,k,dims=(-2,-1)); ob = torch.rot90(ob,k,dims=(-2,-1))
                    if fl == 1:
                        xb, bb, ob = torch.flip(xb,(-1,)), torch.flip(bb,(-1,)), torch.flip(ob,(-1,))
                    elif fl == 2:
                        xb, bb, ob = torch.flip(xb,(-2,)), torch.flip(bb,(-2,)), torch.flip(ob,(-2,))
                logit, elogit = net(xb)
                if cfg['mask_bits']:
                    vis  = 1.0 - ob
                    loss = (F.binary_cross_entropy_with_logits(logit, bb, reduction='none')*vis
                            ).sum()/vis.sum().clamp(min=1)
                else:
                    loss = F.binary_cross_entropy_with_logits(logit, bb)
                if cfg['out_ch'] == 2:
                    loss = loss + F.binary_cross_entropy_with_logits(elogit, ob)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                tot = tot + loss.detach()*len(b); n += len(b)
        return (tot/n).item()            # one sync per epoch, not one per batch

    best, bad, t0, state = 1e9, 0, time.time(), None
    for ep in range(epochs):
        run(tr, True); vl = run(va, False); sch.step(vl)
        if vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            if save: torch.save(state, path)
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval()
    return net, best, time.time()-t0

# ----- timing probe: one short run before committing to all of them
_, vl, secs = train_arm('B', SEEDS[0], epochs=3, patience=99, save=False)
per_epoch = secs/3
print(f'\nprobe: 3 epochs of arm B took {secs:.1f}s, so {per_epoch:.2f}s per epoch')
print(f'projected: about {per_epoch*150/60:.1f} min per run, '
      f'{per_epoch*150*len(ARMS)*len(SEEDS)/3600:.1f} hours for all '
      f'{len(ARMS)*len(SEEDS)} runs')
assert per_epoch < 5, ('still too slow, stop and check the device before continuing')

  set 127: 66 MB of crops moved to cuda
  set random: 66 MB of crops moved to cuda

probe: 3 epochs of arm B took 3.5s, so 1.18s per epoch
projected: about 2.9 min per run, 1.5 hours for all 30 runs


In [10]:
MODELS, TIMES = {}, {}
total = len(ARMS)*len(SEEDS); k = 0
t_all = time.time()
for arm in ARMS:
    for s in SEEDS:
        k += 1
        net, vl, secs = train_arm(arm, s)
        MODELS[(arm,s)] = net; TIMES[(arm,s)] = secs
        tag = 'loaded from checkpoint' if vl is None else f'val {vl:.4f} ({secs:.0f}s)'
        print(f'  [{k:>2}/{total}] arm {arm} seed {s}: {tag}', flush=True)
print(f'\nall {total} runs done in {(time.time()-t_all)/60:.1f} min')

  [ 1/30] arm B seed 0: loaded from checkpoint
  [ 2/30] arm B seed 1: loaded from checkpoint
  [ 3/30] arm B seed 2: loaded from checkpoint
  [ 4/30] arm B seed 3: loaded from checkpoint
  [ 5/30] arm B seed 4: loaded from checkpoint
  [ 6/30] arm C seed 0: loaded from checkpoint
  [ 7/30] arm C seed 1: loaded from checkpoint
  [ 8/30] arm C seed 2: loaded from checkpoint
  [ 9/30] arm C seed 3: loaded from checkpoint
  [10/30] arm C seed 4: loaded from checkpoint
  [11/30] arm D seed 0: loaded from checkpoint
  [12/30] arm D seed 1: loaded from checkpoint
  [13/30] arm D seed 2: loaded from checkpoint
  [14/30] arm D seed 3: loaded from checkpoint
  [15/30] arm D seed 4: loaded from checkpoint
  [16/30] arm E seed 0: loaded from checkpoint
  [17/30] arm E seed 1: loaded from checkpoint
  [18/30] arm E seed 2: loaded from checkpoint
  [19/30] arm E seed 3: loaded from checkpoint
  [20/30] arm E seed 4: loaded from checkpoint
  [21/30] arm M seed 0: loaded from checkpoint
  [22/30] arm

In [11]:
TARGETS  = [3, 6, 9, 12, 15, 18]
COLOURS  = ['grey', 'white', 'black']
POSITION = 'edge'

cal = []
for vid in DEVV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break
def mean_cells(level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, POSITION, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)
curve = [(float(L), mean_cells(float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR = {T: min(curve, key=lambda lc: abs(lc[1]-T))[0] for T in TARGETS}
print('severity calibration:', {T: round(LEVEL_FOR[T],2) for T in TARGETS})

EV = {(c,T): {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]} for c in COLOURS for T in TARGETS}
t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            for T in TARGETS:
                box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    if x32 is None: continue
                    d = EV[(col,T)]
                    d['x'].append((x32*255).astype(np.uint8)); d['id'].append(mid)
                    d['frame'].append(key); d['occ'].append(int(occ.sum())); d['mask'].append(occ)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in EV: EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
n = len(EV[(COLOURS[0],TARGETS[0])]['id'])
print(f'\neval set: {n} markers per condition')
assert n > 1500, f'only {n} markers, data did not load correctly'

severity calibration: {3: 0.24, 6: 0.3, 9: 0.34, 12: 0.4, 15: 0.46, 18: 0.5}
  video_1 (45s)
  video_2 (80s)
  video_3 (124s)
  video_4 (166s)
  video_5 (200s)

eval set: 12276 markers per condition


In [12]:
ACC = {}            # (arm, seed, colour, target) -> per marker correctness
for (arm, s), net in MODELS.items():
    for col in COLOURS:
        for T in TARGETS:
            d = EV[(col,T)]
            P, _ = decode_torch(net, d['x'])
            ACC[(arm,s,col,T)] = (dec_soft(P) == d['id'])
print('evaluated', len(ACC), 'arm and condition combinations\n')

def summary(arm, col, T):
    v = np.array([ACC[(arm,s,col,T)].mean() for s in SEEDS])*100
    return v.mean(), v.std(ddof=1)

print('ARM ACCURACY, mean and sd over', len(SEEDS), 'seeds, development data\n')
for col in COLOURS:
    print(f'--- occluder brightness: {col}')
    print(f"{'cells':>7}" + ''.join(f'{a:>16}' for a in ARMS))
    for T in TARGETS:
        line = f'{EV[(col,T)]["occ"].mean():>7.1f}'
        for arm in ARMS:
            m, sd = summary(arm, col, T); line += f'{m:>10.2f}+-{sd:<4.2f}'
        print(line)
    print()

evaluated 540 arm and condition combinations

ARM ACCURACY, mean and sd over 5 seeds, development data

--- occluder brightness: grey
  cells               B               C               D               E               M               G
    3.2     96.63+-2.74     88.63+-4.58     95.35+-2.01     97.55+-1.60     97.04+-2.15     98.83+-0.23
    5.7     94.36+-3.49     79.34+-7.37     91.75+-2.16     96.15+-2.03     94.70+-2.77     96.49+-1.10
    8.5     91.73+-4.07     72.99+-8.72     89.66+-3.14     94.15+-2.54     92.15+-3.30     94.80+-0.86
   11.5     89.28+-4.32     63.62+-8.96     86.86+-3.19     91.94+-3.41     89.76+-3.16     92.20+-1.71
   15.6     84.75+-3.94     52.95+-10.43     82.23+-3.36     87.75+-3.23     85.33+-2.79     87.55+-1.87
   18.0     79.44+-4.03     41.46+-10.26     76.49+-3.82     82.75+-4.02     79.87+-3.30     80.77+-2.55

--- occluder brightness: white
  cells               B               C               D               E               M               G


## Checkpoint check

B and M are trained with identical settings and seeds. If they disagree by more than about
1 point, or B was saved much earlier than M, some checkpoints came from older code. In that
case delete `ckpt_n13`, rerun from the top, and do not trust the cells below until then.


In [13]:
print('B minus M, per seed, accuracy points (soft decoding)\n')
worst = 0.0
for col in COLOURS:
    for T in TARGETS:
        g = [100*(ACC[('B',s,col,T)].mean() - ACC[('M',s,col,T)].mean()) for s in SEEDS]
        worst = max(worst, max(abs(x) for x in g))
        print(f'{col:>6}{EV[(col,T)]["occ"].mean():>6.1f}  ' + '  '.join(f'{x:>6.2f}' for x in g))

print('\nlargest weight difference between B and M of the same seed:')
for s in SEEDS:
    sb, sm = MODELS[('B',s)].state_dict(), MODELS[('M',s)].state_dict()
    diff = max(float((sb[k]-sm[k]).abs().max()) for k in sb)
    tb = time.ctime(os.path.getmtime(f'{CKPT}/arm_B_s{s}.pt'))
    tm = time.ctime(os.path.getmtime(f'{CKPT}/arm_M_s{s}.pt'))
    print(f'  seed {s}: {diff:.2e}   B saved {tb}   M saved {tm}')

print(f'\nlargest B minus M gap: {worst:.2f} points')
print('OK, checkpoints look consistent' if worst < 1.0 else
      'WARNING: same settings give different results. If B was saved much earlier than M, '
      'B came from the old training code. Delete ckpt_n13 and retrain all 30 runs.')


B minus M, per seed, accuracy points (soft decoding)

  grey   3.2   -1.44   -1.12    0.00    0.28    0.25
  grey   5.7   -1.65   -1.03   -0.11    0.59    0.55
  grey   8.5   -2.00   -1.63    0.30    0.56    0.62
  grey  11.5   -1.72   -2.48   -0.49    0.97    1.29
  grey  15.6   -2.00   -2.33   -0.81    0.81    1.43
  grey  18.0   -0.74   -2.41   -0.59    0.45    1.17
 white   3.2   -1.41   -2.74   -0.17    1.47    1.16
 white   5.7   -1.74   -1.87   -0.31    2.05    1.90
 white   8.5   -1.73   -3.38    0.22    1.60    1.30
 white  11.5   -1.52   -3.45   -0.89    1.84    2.87
 white  15.6   -1.67   -2.96   -1.03    1.44    1.70
 white  18.0   -0.90   -3.28   -0.51    1.05    2.00
 black   3.2   -0.62   -1.94   -0.63   -0.04   -0.10
 black   5.7   -0.30   -3.36   -0.66    0.27   -0.41
 black   8.5   -0.57   -2.35   -0.42   -0.37   -0.16
 black  11.5    0.16   -2.92   -2.61    0.36    0.00
 black  15.6    0.03   -2.97   -2.57   -0.07    0.30
 black  18.0    0.59   -1.60   -1.79    0.14 

## Unified evaluation

Every arm plus the published decoder (P), on grey, white and black, with these decoders:

* `soft`: soft log likelihood, as in the cell above
* `hard`: nearest codeword with exact fair tie breaking (expected accuracy, no random draws)
* `soft_erasure`: soft decoding weighted by 1 minus the erasure output (C and D only)
* `soft_oracle`, `hard_oracle`: hidden cells removed using the true mask, an upper bound

The last block prints grey minus black per arm. Positive means black is harder.


In [14]:
def hamming_w(P, W=None):
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    W = np.ones_like(B) if W is None else W.reshape(len(P),36).astype(np.float32)
    return np.stack([(B*W)@(1-ROT[k]).T + ((1-B)*W)@ROT[k].T for k in range(4)]).min(0)

def hard_expected(P, ids, W=None):
    """Exact expected accuracy with fair random tie breaking, per marker (0 to 1)."""
    D = hamming_w(P, W); tied = D == D.min(1, keepdims=True)
    return tied[np.arange(len(ids)), ids] / tied.sum(1)

DECODERS = ['soft', 'hard', 'soft_erasure', 'soft_oracle', 'hard_oracle']

def score(P, E, d):
    ids = d['id']; vis = 1.0 - d['mask'].astype(np.float32)
    out = {'soft':        (dec_soft(P) == ids).astype(np.float32),
           'hard':        hard_expected(P, ids),
           'soft_oracle': (dec_soft(P, vis) == ids).astype(np.float32),
           'hard_oracle': hard_expected(P, ids, vis)}
    if E is not None:
        out['soft_erasure'] = (dec_soft(P, 1 - E) == ids).astype(np.float32)
    return out

R = {}; t0 = time.time()
for k, d in EV.items():
    for dec, v in score(decode_published(d['x']), None, d).items():
        R[('P','pub') + k + (dec,)] = v
for (arm, s), net in MODELS.items():
    for k, d in EV.items():
        P, E = decode_torch(net, d['x'])
        for dec, v in score(P, E if ARMS[arm]['out_ch'] == 2 else None, d).items():
            R[(arm, s) + k + (dec,)] = v
print(f'scored {len(R)} combinations in {time.time()-t0:.0f}s\n')

ARM_SEEDS = {'P': ['pub'], **{a: SEEDS for a in ARMS}}
def stat(arm, col, T, dec):
    v = [100*R[(arm,s,col,T,dec)].mean() for s in ARM_SEEDS[arm] if (arm,s,col,T,dec) in R]
    if not v: return None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

print('ACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)\n')
for col in COLOURS:
    print(f'--- {col}')
    print(f"{'arm':>4}{'decoder':>14}" + ''.join(f'{EV[(col,T)]["occ"].mean():>13.1f}' for T in TARGETS))
    for arm in ARM_SEEDS:
        for dec in DECODERS:
            if stat(arm, col, TARGETS[0], dec) is None: continue
            print(f'{arm:>4}{dec:>14}' + ''.join(
                f'{m:>8.2f}+-{sd:<3.1f}' for m, sd in (stat(arm, col, T, dec) for T in TARGETS)))
        print()

print('BLACK INVERSION: grey minus black accuracy, positive means black is harder\n')
for dec in ['soft', 'hard']:
    print(f'--- {dec} decoding')
    for arm in ARM_SEEDS:
        print(f'{arm:>4}' + ''.join(f'{stat(arm,"grey",T,dec)[0]-stat(arm,"black",T,dec)[0]:>9.2f}'
                                    for T in TARGETS))
    print()

out = {f'{a}|{s}|{c}|{T}|{dec}': float(v.mean()) for (a,s,c,T,dec), v in R.items()}
with open(f'{ART}/n13_unified_dev.json','w') as fh: json.dump(out, fh, indent=1)
print('saved', f'{ART}/n13_unified_dev.json')


scored 2412 combinations in 327s

ACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)

--- grey
 arm       decoder          3.2          5.7          8.5         11.5         15.6         18.0
   P          soft   85.70+-0.0   76.33+-0.0   65.40+-0.0   49.22+-0.0   31.97+-0.0   22.59+-0.0
   P          hard   89.03+-0.0   79.43+-0.0   69.45+-0.0   52.54+-0.0   34.67+-0.0   23.35+-0.0
   P   soft_oracle   93.16+-0.0   86.19+-0.0   78.51+-0.0   66.97+-0.0   54.37+-0.0   43.61+-0.0
   P   hard_oracle   94.84+-0.0   88.22+-0.0   81.69+-0.0   69.44+-0.0   56.35+-0.0   45.65+-0.0

   B          soft   96.63+-2.7   94.36+-3.5   91.73+-4.1   89.28+-4.3   84.75+-3.9   79.44+-4.0
   B          hard   96.37+-3.1   92.90+-2.9   87.25+-3.6   75.44+-2.8   55.11+-1.6   36.45+-2.0
   B   soft_oracle   97.13+-2.6   95.34+-3.1   93.65+-4.2   90.19+-4.2   87.14+-4.5   80.17+-4.5
   B   hard_oracle   97.26+-2.9   96.09+-2.9   94.84+-3.4   90.15+-3.3   84.86+-3.3   75.2

In [15]:
def seed_diff(a1, a2, col, T):
    v = np.array([ (ACC[(a1,s,col,T)].mean() - ACC[(a2,s,col,T)].mean()) for s in SEEDS ])*100
    m, sd = v.mean(), v.std(ddof=1)
    tcrit = {2:12.706, 3:4.303, 4:3.182, 5:2.776, 6:2.571, 7:2.447}.get(len(SEEDS), 2.776)
    half = tcrit * sd / np.sqrt(len(SEEDS))
    return m, sd, m-half, m+half

PAIRS = [('D','B','does the erasure output help'),
         ('C','D','does masking the loss on hidden cells help'),
         ('E','B','does random brightness training help'),
         ('M','G','is max pooling the cause')]

print(f"{'pair':>7}{'col':>7}{'cells':>7}{'mean diff':>11}{'sd':>7}{'95% CI over seeds':>22}  verdict")
VERD = []
for a1, a2, q in PAIRS:
    for col in COLOURS:
        for T in TARGETS:
            m, sd, lo, hi = seed_diff(a1, a2, col, T)
            if lo > 0 or hi < 0:                 v = 'differs'
            elif lo > -MARGIN and hi < MARGIN:   v = f'equivalent (within {MARGIN})'
            else:                                v = 'inconclusive'
            VERD.append(dict(pair=f'{a1}-{a2}', colour=col, target=T,
                             cells=float(EV[(col,T)]['occ'].mean()),
                             mean=float(m), sd=float(sd), lo=float(lo), hi=float(hi), verdict=v))
            print(f'{a1+"-"+a2:>7}{col:>7}{EV[(col,T)]["occ"].mean():>7.1f}'
                  f'{m:>11.2f}{sd:>7.2f}   [{lo:>6.2f},{hi:>6.2f}]  {v}')
    print()
print('A claim of no effect is only allowed where the verdict says equivalent.')

   pair    col  cells  mean diff     sd     95% CI over seeds  verdict
    D-B   grey    3.2      -1.28   2.33   [ -4.17,  1.61]  inconclusive
    D-B   grey    5.7      -2.61   3.51   [ -6.97,  1.74]  inconclusive
    D-B   grey    8.5      -2.06   4.32   [ -7.43,  3.30]  inconclusive
    D-B   grey   11.5      -2.41   4.31   [ -7.77,  2.94]  inconclusive
    D-B   grey   15.6      -2.53   4.13   [ -7.65,  2.60]  inconclusive
    D-B   grey   18.0      -2.95   4.84   [ -8.96,  3.07]  inconclusive
    D-B  white    3.2      -3.20   4.29   [ -8.53,  2.12]  inconclusive
    D-B  white    5.7      -3.94   5.23   [-10.43,  2.55]  inconclusive
    D-B  white    8.5      -2.98   5.12   [ -9.34,  3.38]  inconclusive
    D-B  white   11.5      -3.37   5.15   [ -9.77,  3.02]  inconclusive
    D-B  white   15.6      -2.86   4.66   [ -8.64,  2.92]  inconclusive
    D-B  white   18.0      -3.73   6.04   [-11.23,  3.77]  inconclusive
    D-B  black    3.2      -2.97   2.61   [ -6.21,  0.27]  inconc

In [16]:
print('C AGAINST D AT EVERY SEVERITY, the mixed result\n')
print(f"{'col':>7}{'cells':>7}{'C':>9}{'D':>9}{'D - C':>9}{'95% CI over seeds':>22}")
cd = []
for col in COLOURS:
    for T in TARGETS:
        mc = np.mean([ACC[('C',s,col,T)].mean() for s in SEEDS])*100
        md = np.mean([ACC[('D',s,col,T)].mean() for s in SEEDS])*100
        m, sd, lo, hi = seed_diff('D','C',col,T)
        cd.append(dict(colour=col, target=T, C=float(mc), D=float(md),
                       diff=float(m), lo=float(lo), hi=float(hi)))
        print(f'{col:>7}{EV[(col,T)]["occ"].mean():>7.1f}{mc:>9.2f}{md:>9.2f}{m:>9.2f}'
              f'   [{lo:>6.2f},{hi:>6.2f}]')
wins_d = sum(1 for r in cd if r['lo'] > 0); wins_c = sum(1 for r in cd if r['hi'] < 0)
print(f'\nD clearly better in {wins_d} of {len(cd)} conditions, C clearly better in {wins_c}.')
print('Write this as a mixed result if both counts are non zero.')

res = dict(seeds=SEEDS, margin=MARGIN,
           arms={k: {kk: str(vv) for kk,vv in v.items()} for k,v in ARMS.items()},
           levels={str(k): v for k,v in LEVEL_FOR.items()},
           seconds={f'{a}|{s}': float(t) for (a,s),t in TIMES.items()},
           accuracy={f'{a}|{s}|{c}|{T}': float(ACC[(a,s,c,T)].mean()) for (a,s,c,T) in ACC},
           equivalence=VERD, c_vs_d=cd)
with open(f'{ART}/n13_seeds_dev.json','w') as fh: json.dump(res, fh, indent=1)
print('\nsaved', f'{ART}/n13_seeds_dev.json')

C AGAINST D AT EVERY SEVERITY, the mixed result

    col  cells        C        D    D - C     95% CI over seeds
   grey    3.2    88.63    95.35     6.72   [  1.66, 11.79]
   grey    5.7    79.34    91.75    12.41   [  4.07, 20.75]
   grey    8.5    72.99    89.66    16.67   [  6.73, 26.61]
   grey   11.5    63.62    86.86    23.24   [ 13.39, 33.09]
   grey   15.6    52.95    82.23    29.27   [ 18.10, 40.45]
   grey   18.0    41.46    76.49    35.04   [ 23.71, 46.37]
  white    3.2    78.82    86.51     7.69   [  1.19, 14.19]
  white    5.7    66.47    81.61    15.14   [  5.25, 25.02]
  white    8.5    57.64    79.06    21.42   [  8.57, 34.27]
  white   11.5    45.44    76.09    30.65   [ 15.99, 45.32]
  white   15.6    33.38    71.85    38.47   [ 22.60, 54.34]
  white   18.0    23.95    65.37    41.42   [ 26.49, 56.34]
  black    3.2    93.85    85.92    -7.92   [-10.70, -5.15]
  black    5.7    88.00    81.16    -6.83   [ -8.70, -4.96]
  black    8.5    77.59    71.53    -6.05   [-1